In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os


In [5]:
df = pd.read_csv(r'C:\Users\Hp\Documents\coolyeah\sems 4\PASD\tubes2\data\Movie Dataset.csv')

In [6]:
# cek missing value
print(df.isnull().sum())

Release_Date          0
Title                 9
Overview              9
Popularity           10
Vote_Count           10
Vote_Average         10
Original_Language    10
Genre                11
Poster_Url           11
dtype: int64


In [7]:
# cleaning data

df_clean = df.copy()

# Hapus baris dengan missing value di kolom penting
df_clean = df_clean.dropna(subset=['Title', 'Genre', 'Vote_Average'])

# Isi Popularity yang kosong dengan median
df_clean['Popularity'] = pd.to_numeric(df_clean['Popularity'], errors='coerce')
df_clean['Popularity'] = df_clean['Popularity'].fillna(df_clean['Popularity'].median())

# Isi Overview yang kosong dengan string kosong
df_clean['Overview'] = df_clean['Overview'].fillna('').astype(str)

# Ekstrak tahun dari Release_Date
df_clean['Year'] = pd.to_datetime(df_clean['Release_Date'], errors='coerce').dt.year

# Hapus baris yang tahunnya tidak bisa diekstrak
df_clean = df_clean.dropna(subset=['Year'])
df_clean['Year'] = df_clean['Year'].astype(int)

# Pastikan Vote_Average adalah numeric
df_clean['Vote_Average'] = pd.to_numeric(df_clean['Vote_Average'], errors='coerce')
df_clean = df_clean.dropna(subset=['Vote_Average'])

print(f"setelah cleaning: {len(df_clean)} film")
print(f"rentang tahun: {df_clean['Year'].min()} - {df_clean['Year'].max()}")
print(f"tipe data Vote_Average: {df_clean['Vote_Average'].dtype}")

setelah cleaning: 9826 film
rentang tahun: 1902 - 2024
tipe data Vote_Average: float64


one hot encoding

In [8]:
# Pisahkan genre 
df_clean['Genre_List'] = df_clean['Genre'].str.split(', ')

# Ambil semua genre unik
all_genres = set()
for genres in df_clean['Genre_List']:
    all_genres.update(genres)

all_genres = sorted(list(all_genres))
print(f"total genre unik: {len(all_genres)}")
print(f"genre: {all_genres[:15]}...")

# One-Hot Encoding
for genre in all_genres:
    df_clean[f'genre_{genre}'] = df_clean['Genre_List'].apply(
        lambda x: 1 if genre in x else 0
    )


total genre unik: 19
genre: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction']...


fitur untuk model prediksi rating

In [9]:
# Fitur yang akan digunakan
numeric_features = ['Popularity', 'Year']
genre_features = [f'genre_{g}' for g in all_genres]

feature_cols = numeric_features + genre_features

# Siapkan X (fitur) dan y (target)
X = df_clean[feature_cols].copy()
y = df_clean['Vote_Average'].copy()

# Isi nilai NaN jika masih ada
X = X.fillna(X.median())

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Total fitur: {len(feature_cols)}")

X shape: (9826, 21)
y shape: (9826,)
Total fitur: 21


In [10]:
# Normalisasi fitur numerik
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"data training: {len(X_train)} film")
print(f"data testing: {len(X_test)} film")

data training: 7860 film
data testing: 1966 film


latih model random forest

In [11]:
# Latih model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Model Random Forest selesai dilatih")

Model Random Forest selesai dilatih


In [13]:
# Prediksi pada data test
y_pred = rf_model.predict(X_test)

# Hitung metrik
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Hitung akurasi (prediksi dalam rentang ±0.5)
accuracy = (abs(y_test - y_pred) <= 0.5).mean() * 100

print("EVALUASI MODEL")
print(f"RMSE: {rmse:.3f}")
print(f"R² Score: {r2:.3f}")
print(f"Akurasi (±0.5): {accuracy:.1f}%")

EVALUASI MODEL
RMSE: 0.880
R² Score: 0.291
Akurasi (±0.5): 50.5%


simpan model dan scaler

In [14]:
joblib.dump(rf_model, r'C:\Users\Hp\Documents\coolyeah\sems 4\PASD\tubes2\models\rf_regressor.pkl')
joblib.dump(scaler, r'C:\Users\Hp\Documents\coolyeah\sems 4\PASD\tubes2\models\rf_scaler.pkl')

# Simpan daftar genre untuk frontend
with open(r'C:\Users\Hp\Documents\coolyeah\sems 4\PASD\tubes2\models\genre_list.txt', 'w') as f:
    for genre in all_genres:
        f.write(f"{genre}\n")

# Simpan dataset yang sudah di clean
df_clean.to_csv(r'C:\Users\Hp\Documents\coolyeah\sems 4\PASD\tubes2\data\movies_cleaned.csv', index=False)